In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed
import pyart
import geopy.distance as distance
import haversine


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
#print(P2)

In [3]:
ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP6.nc')

In [4]:
location = ncfile1.variables['location'][:]
qc = ncfile1.variables['qc'][:]
obstype = ncfile1.variables['obs_type'][:]
obstypemd = ncfile1.variables['ObsTypesMetaData'][:]
obs_val = ncfile1.variables['observations'][:]
which_vert = ncfile1.variables['which_vert'][:]
times = ncfile1.variables['time'][:]
print(obstype)
qc_new = []
for i in range(len(qc)):
    qc_d = qc[i][0]
    qc_new.append(qc_d)
qc_new = np.asarray(qc_new)

[142 105 106 ... 127  27 127]


In [5]:
print(np.unique(obstype))

[ 16  17  18  25  26  27  28 105 106 107 108 121 127 141 142]


In [6]:
otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []

minute_range = np.arange(180,185,5)
#minute_range = np.arange(595,605,5)

for mins in minute_range:
    dt_start = datetime(2022,9,15,0,0)
    #dt_start = datetime(2022,7,19,0,0)
    #dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    #dt = datetime(2022,9,15,9,55)
    wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    
    
    #otype = 107
    #otype = 105
    #otype = 106
    #otype = 108
    #otype = 142
    otype_list = [25,26,27,28]
    for otype in otype_list:
        loc_T2 = location[obstype==otype, :]
        qc_T2 = qc_new[obstype==otype]
        obs_T2 = obs_val[obstype==otype, :]
        lons_T2 = loc_T2[:,0]
        lats_T2 = loc_T2[:,1]
        elev_T2 = loc_T2[:,2]
        time_T2 = times[obstype==otype]
        lons_T2[lons_T2 > 180] = lons_T2[lons_T2 > 180] - 360
        
        #Convert WRF file time into same units as the obs_seq time
        dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400
        time_diff = np.abs(dt_tot - time_T2)
        
        #Get obs within +- 2.5 minutes of each WRF file
        time_T3 = time_T2[time_diff<(150/86400)]
        loc_T3 = loc_T2[time_diff<(150/86400), :]
        qc_T3 = qc_T2[time_diff<(150/86400)]
        lons_T3 = lons_T2[time_diff<(150/86400)]
        lats_T3 = lats_T2[time_diff<(150/86400)]
        elev_T3 = elev_T2[time_diff<(150/86400)]
        obs_T3 = obs_T2[time_diff<(150/86400), :]
        
        if len(obs_T3)==0:
            print('NO OBS IN WINDOW')
        for k in range(len(lons_T3)):
            latp=lats_T3[k]
            lonp=lons_T3[k]
            #Get location for each ob in model land
            lon1d = np.ndarray.flatten(lon[0,:,:])
            lat1d = np.ndarray.flatten(lat[0,:,:])
            station = []
            points = []
            for i in range(len(lon1d)):
                points.append((lat1d[i],lon1d[i]))
                station.append((latp,lonp))
            dist = haversine.haversine_vector(station,points)
            dist2=dist.reshape(lon.shape[1],lon.shape[2])
            print(lon[0,:,:][np.where(dist2==np.min(dist2))])
            print(lat[0,:,:][np.where(dist2==np.min(dist2))])
            print(np.where(dist2==np.min(dist2)), dt)
            st_xind = np.where(dist2==np.min(dist2))[0][0]
            st_yind = np.where(dist2==np.min(dist2))[1][0]
            if otype == 27:
                T2_a = T2[0,st_xind,st_yind].magnitude
            elif otype == 25:
                T2_a = U10[0,st_xind,st_yind]
            elif otype == 26:
                T2_a = V10[0,st_xind,st_yind]
            elif otype == 28:
                T2_a = Q2[0,st_xind,st_yind]
            #If you want to change the error assumption, just change the scale in this line
            error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[k,1]))
            #6/17/2024 MW adding code to limit added error to 1.5 times the error assumtion in DART
            if np.abs(error/4) > (np.sqrt(obs_T3[k,1])*1.0):
                error = (error / np.abs(error)) * (np.sqrt(obs_T3[k,1])*1.0)
            T2_b = T2_a + (error/4)
            print(T2_a, (error/4))
        
            #Append obs to arrays for writing to obs_seq file later
            otype_s.append(otype)
            obs_s.append(T2_b)
            lon_s.append(lonp)
            lat_s.append(latp)
            elev_s.append(elev_T3[k])
            error_s.append(obs_T3[k,1]/2)
            time_s.append(time_T3[k])

2022-09-15 03:00:00
[-84.49754]
[39.095318]
(array([426]), array([725])) 2022-09-15 03:00:00
0.21833003 1.5270508609415347
[-84.06508]
[39.64163]
(array([976]), array([1054])) 2022-09-15 03:00:00
-2.043291 -1.118386861491902
[-85.13965]
[39.641056]
(array([970]), array([226])) 2022-09-15 03:00:00
2.0178297 1.0833684614824863
[-84.76526]
[38.90871]
(array([238]), array([518])) 2022-09-15 03:00:00
-1.7109983 0.5295645653912916
[-84.79973]
[39.399544]
(array([729]), array([489])) 2022-09-15 03:00:00
0.159646 0.10741028026274095
[-84.22902]
[39.424213]
(array([757]), array([930])) 2022-09-15 03:00:00
-2.266208 1.0228393256657862
[-84.85171]
[39.089905]
(array([419]), array([450])) 2022-09-15 03:00:00
0.26193938 1.179132606104198
[-84.38455]
[39.289467]
(array([621]), array([811])) 2022-09-15 03:00:00
-0.7923241 -0.7630717797488874
[-84.6622]
[38.948265]
(array([278]), array([598])) 2022-09-15 03:00:00
-1.4494514 -0.02923249204364155
[-84.40468]
[38.994774]
(array([326]), array([798])) 2022

In [7]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
day_DART = 154024
#day_DART = 153966
#day_DART = 153556

seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [8]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
# print(seconds_DART)
# print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]

In [9]:
for bigfoot in [1,2]:
    print(bigfoot)

    #Write the simulated obs out to an obs_seq file
    filename = 'SIM_MESONET03_IOP6_z'
    #filename = 'SIM_MESONET_IOP4_halferr'
    #filename = 'SIM_MESONET_NEW_halferr'
    
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(4))
    fi.write("    %d          %s   \n" %(27, 'LAND_SFC_TEMPERATURE'))
    fi.write("    %d          %s   \n" %(25, 'LAND_SFC_U_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(26, 'LAND_SFC_V_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(28, 'LAND_SFC_SPECIFIC_HUMIDITY'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s1)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s1):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], 3))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


In [10]:
print(obs_T3[:,1])

#Get a randomly-generated error value to add to the synthetic ob 
#by sampling a Gaussian distribution with the same standard deviation as
#the assumed error from DART
error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[0,1]))
print(error)

[5.34578267e-06 8.18482181e-06 6.68662239e-06 6.28555680e-06
 6.68342873e-06 6.23898951e-06 6.67539981e-06 6.31682179e-06
 5.80385296e-06 4.36343869e-06 5.43163189e-06 6.68058745e-06
 5.87715289e-06 5.45748257e-06 5.79926104e-06 6.10569428e-06
 7.10038477e-06 6.23972851e-06 5.49598216e-06 5.44538800e-06
 6.25377527e-06 6.15849463e-06 8.32583054e-06 6.68171361e-06
 6.74459127e-06 6.25302895e-06 6.75111349e-06 5.36693164e-06
 6.22575600e-06 6.29563978e-06 6.24060770e-06 5.11083208e-06
 5.81514481e-06 7.18242433e-06 6.61412287e-06 6.70587487e-06
 6.24938495e-06 5.79016917e-06 5.88845441e-06 6.82292990e-06
 6.75335265e-06 7.74460359e-06 6.72831021e-06 6.19099776e-06
 5.81438453e-06 8.32904509e-06 5.83497017e-06 6.25328969e-06
 7.61831352e-06]
0.0018284962398222822


In [11]:
print(len(obs_s))

303


In [12]:
#Convert WRF file time into same units as the obs_seq time
dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400

time_diff = np.abs(dt_tot - time_T2)

print(time_diff[time_diff<(150/86400)])

print(dt_tot)
print(time_diff)

[0.0013541666558012366 0.0011805555550381541 0.0011458333465270698
 0.0011458333465270698 0.0010879629699047655 0.0009606481471564621
 0.0007870370463933796 0.0007638888782821596 0.0006944444321561605
 0.00048611112288199365 0.00048611112288199365 0.00040509257814846933
 0.0003819444391410798 0.0001273148227483034 4.629630711860955e-05 0.0
 4.629630711860955e-05 4.629630711860955e-05 4.629630711860955e-05
 5.787037662230432e-05 6.944444612599909e-05 9.259258513338864e-05
 0.00011574075324460864 0.00015046296175569296 0.0001736111007630825
 0.00021990740788169205 0.0003124999930150807 0.00035879630013369024
 0.000370370369637385 0.0003819444391410798 0.00042824074625968933
 0.00042824074625968933 0.0004513888852670789 0.0005439814704004675
 0.0005902777775190771 0.0006018518470227718 0.0006134259165264666
 0.0006249999860301614 0.0006365740846376866 0.0006481481541413814
 0.0006712962931487709 0.0006828703626524657 0.0007175926002673805
 0.0007754629768896848 0.0007870370463933796 0.000

In [13]:
print(dt.isoformat()[14:16])

00


In [14]:
print(len(obs_s))

303
